In [0]:
%python
dbutils.widgets.text("catalog", "tpch_sales")
catalog = dbutils.widgets.get("catalog")

In [0]:
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.dim_geography')
COMMENT 'Geography dimension – one row per nation with its parent region'
AS
SELECT
    n.nationkey AS nation_key,
    n.name AS nation_name,
    r.regionkey AS region_key,
    r.name AS region_name
FROM IDENTIFIER(:catalog || '.silver.nation')  n
JOIN IDENTIFIER(:catalog || '.silver.region') r
    ON n.regionkey = r.regionkey;

In [0]:
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.dim_customer')
COMMENT 'Customer dimension – one row per customer'
AS
SELECT
    c.custkey AS customer_key,
    c.name AS customer_name,
    c.address AS customer_address,
    c.phone AS customer_phone,
    c.acctbal AS account_balance,
    c.mktsegment AS market_segment,
    g.nation_key  AS customer_nation_key,
    g.nation_name AS customer_nation_name,
    g.region_key AS customer_region_key,
    g.region_name AS customer_region_name
FROM IDENTIFIER(:catalog || '.silver.customer')  c
JOIN IDENTIFIER(:catalog || '.gold.dim_geography') g
    ON c.nationkey = g.nation_key;


In [0]:
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.dim_supplier')
COMMENT 'Supplier dimension – one row per supplier'
AS
SELECT 
    s.suppkey AS supplier_key,
    s.name AS supplier_name,
    s.address AS supplier_address,
    s.phone AS supplier_phone,
    s.acctbal AS account_balance,
    g.nation_key  AS supplier_nation_key,
    g.nation_name AS supplier_nation_name,
    g.region_key AS supplier_region_key,
    g.region_name AS supplier_region_name
FROM IDENTIFIER(:catalog || '.silver.supplier')  s
JOIN IDENTIFIER(:catalog || '.gold.dim_geography') g
    ON s.nationkey = g.nation_key;

In [0]:
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.dim_product')
COMMENT 'Product dimension – one row per product'
AS
SELECT
    p.partkey AS product_key,
    p.name AS product_name,
    p.mfgr AS manufacturer,
    p.brand AS brand, 
    p.type AS product_type,
    p.size AS size,
    p.container AS container,
    p.retailprice AS retail_price
FROM IDENTIFIER(:catalog || '.silver.part') p;


In [0]:
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.fact_sales')
COMMENT 'Sales fact table – one row per order line'
AS
SELECT
    l.orderkey AS order_key,
    l.linenumber AS line_number,
    c.customer_key AS customer_key,
    s.supplier_key AS supplier_key,
    p.product_key AS product_key,
    o.orderdate AS order_date,
    o.orderstatus AS order_status,
    o.orderpriority AS order_priority,
    o.totalprice AS order_total_price,
    l.quantity AS quantity,
    l.extendedprice AS extended_price,
    l.discount AS discount,
    l.tax AS tax,
    l.returnflag AS return_flag,
    l.linestatus AS line_status,
    l.shipdate AS ship_date,
    l.commitdate AS commit_date,
    l.receiptdate AS receipt_date,
    l.shipmode AS ship_mode,
    c.customer_nation_key,
    c.customer_nation_name,
    c.customer_region_key,
    c.customer_region_name,
    s.supplier_nation_key,
    s.supplier_nation_name,
    s.supplier_region_key,
    s.supplier_region_name
FROM IDENTIFIER(:catalog || '.silver.lineitem') l
JOIN IDENTIFIER(:catalog || '.silver.orders') o
    ON l.orderkey = o.orderkey
JOIN IDENTIFIER(:catalog || '.gold.dim_customer') c
    ON o.custkey = c.customer_key
JOIN IDENTIFIER(:catalog || '.gold.dim_supplier') s
    ON l.suppkey = s.supplier_key
JOIN IDENTIFIER(:catalog || '.gold.dim_product') p
    ON l.partkey = p.product_key;


In [0]:
CREATE OR REPLACE VIEW IDENTIFIER(:catalog || '.gold.vw_annual_sales_kpi')
COMMENT 'Annual sales KPIs – revenue, orders, active customers, AOV'
AS
SELECT
    YEAR(order_date) AS sales_year,
    SUM(extended_price * (1 - discount)) AS net_revenue,
    COUNT(DISTINCT order_key) AS orders,
    COUNT(DISTINCT customer_key) AS active_customers,
    SUM(extended_price * (1 - discount)) / COUNT(DISTINCT order_key) AS average_order_value
FROM IDENTIFIER(:catalog || '.gold.fact_sales')
GROUP BY YEAR(order_date);

In [0]:
-- Gold fact_sales vs independent Silver calculation
WITH gold_metrics AS (
    SELECT
        COUNT(*) AS gold_row_count,
        SUM(extended_price * (1 - discount)) AS gold_net_revenue
    FROM IDENTIFIER(:catalog || '.gold.fact_sales')
),
silver_metrics AS (
    SELECT
        COUNT(*) AS silver_row_count,
        SUM(l.extendedprice * (1 - l.discount)) AS silver_net_revenue
    FROM IDENTIFIER(:catalog || '.silver.lineitem') l
    JOIN IDENTIFIER(:catalog || '.silver.orders') o
        ON l.orderkey = o.orderkey
    JOIN IDENTIFIER(:catalog || '.silver.customer') c
        ON o.custkey = c.custkey
    JOIN IDENTIFIER(:catalog || '.silver.supplier') s
        ON l.suppkey = s.suppkey
    JOIN IDENTIFIER(:catalog || '.silver.part') p
        ON l.partkey = p.partkey
)
SELECT
    g.gold_row_count,
    s.silver_row_count,
    g.gold_row_count - s.silver_row_count AS row_count_diff,
    ROUND(g.gold_net_revenue, 2) AS gold_net_revenue,
    ROUND(s.silver_net_revenue, 2) AS silver_net_revenue,
    ROUND(g.gold_net_revenue - s.silver_net_revenue, 2) AS revenue_diff,
    CASE
        WHEN g.gold_row_count = s.silver_row_count
         AND g.gold_net_revenue = s.silver_net_revenue
        THEN 'PASS'
        ELSE 'FAIL'
    END AS status
FROM gold_metrics g
CROSS JOIN silver_metrics s;

In [0]:
-- Dimension Key Uniqueness Validation
SELECT 'dim_geography' AS table_name, 'nation_key' AS key_column,
       COUNT(*) AS total_rows, COUNT(DISTINCT nation_key) AS distinct_keys,
       CASE WHEN COUNT(*) = COUNT(DISTINCT nation_key) THEN 'PASS' ELSE 'FAIL' END AS status
FROM IDENTIFIER(:catalog || '.gold.dim_geography')
UNION ALL
SELECT 'dim_customer', 'customer_key',
       COUNT(*), COUNT(DISTINCT customer_key),
       CASE WHEN COUNT(*) = COUNT(DISTINCT customer_key) THEN 'PASS' ELSE 'FAIL' END
FROM IDENTIFIER(:catalog || '.gold.dim_customer')
UNION ALL
SELECT 'dim_supplier', 'supplier_key',
       COUNT(*), COUNT(DISTINCT supplier_key),
       CASE WHEN COUNT(*) = COUNT(DISTINCT supplier_key) THEN 'PASS' ELSE 'FAIL' END
FROM IDENTIFIER(:catalog || '.gold.dim_supplier')
UNION ALL
SELECT 'dim_product', 'product_key',
       COUNT(*), COUNT(DISTINCT product_key),
       CASE WHEN COUNT(*) = COUNT(DISTINCT product_key) THEN 'PASS' ELSE 'FAIL' END
FROM IDENTIFIER(:catalog || '.gold.dim_product');